# Groth16 verify walkthrough

Olympus does **not** reimplement the BN128 pairing check in Python. Instead,
`Groth16Prover.verify()` writes the proof and public signals to JSON and delegates the final
pairing equation to `snarkjs groth16 verify`. This notebook makes that boundary explicit.


The pairing equation checked by Groth16 on BN128 is conceptually:

`e(pi_a, pi_b) == e(alpha_1, beta_2) * e(vk_x, gamma_2) * e(pi_c, delta_2)`

where `vk_x` is the linear combination of the verification-key `IC` points with the public inputs.
Olympus' Python code prepares those public inputs and then shells out to `snarkjs` for the actual
elliptic-curve and pairing arithmetic.


In [ ]:
import json
import subprocess
from pathlib import Path
from unittest.mock import patch

from protocol.zkp import Groth16Prover, ZKProof


In [1]:
repo_root = Path('/home/runner/work/Olympus/Olympus')
vkey_path = repo_root / 'proofs' / 'keys' / 'verification_keys' / 'document_existence_vkey.json'
vkey = json.loads(vkey_path.read_text(encoding='utf-8'))
print(f'verification key path: {vkey_path}')
print(f'protocol: {vkey["protocol"]}')
print(f'curve: {vkey["curve"]}')
print(f'nPublic: {vkey["nPublic"]}')
print(f'len(IC): {len(vkey["IC"])}')
print(f'len(vk_alpha_1): {len(vkey["vk_alpha_1"])}')
print(f'len(vk_beta_2): {len(vkey["vk_beta_2"])}')
print(f'len(vk_gamma_2): {len(vkey["vk_gamma_2"])}')
print(f'len(vk_delta_2): {len(vkey["vk_delta_2"])}')


verification key path: /home/runner/work/Olympus/Olympus/proofs/keys/verification_keys/document_existence_vkey.json
protocol: groth16
curve: bn128
nPublic: 2
len(IC): 3
len(vk_alpha_1): 3
len(vk_beta_2): 3
len(vk_gamma_2): 3
len(vk_delta_2): 3


In [2]:
            print(
                r"""template DocumentExistence(depth) {
    signal input root;
    signal input leafIndex;
    signal input leaf;
    signal input pathElements[depth];
    signal input pathIndices[depth];
    ...
}

component main {public [root, leafIndex]} = DocumentExistence(20);"""
            )


template DocumentExistence(depth) {
    signal input root;
    signal input leafIndex;
    signal input leaf;
    signal input pathElements[depth];
    signal input pathIndices[depth];
    ...
}

component main {public [root, leafIndex]} = DocumentExistence(20);


In [3]:
captured = {}
proof = ZKProof(
    proof={'pi_a': ['1', '2'], 'pi_b': [['3', '4'], ['5', '6']], 'pi_c': ['7', '8']},
    public_signals=['12345', '0'],
    circuit='document_existence',
)
prover = Groth16Prover(repo_root / 'proofs', snarkjs_bin='npx')

def fake_run(cmd, cwd=None, check=None, capture_output=None, text=None):
    captured['cmd'] = cmd
    captured['cwd'] = str(cwd)
    captured['public'] = json.loads(Path(cmd[-2]).read_text(encoding='utf-8'))
    captured['proof'] = json.loads(Path(cmd[-1]).read_text(encoding='utf-8'))
    return subprocess.CompletedProcess(cmd, 0, stdout='OK', stderr='')

with patch('shutil.which', return_value='/usr/bin/npx'), patch('subprocess.run', side_effect=fake_run):
    verified = prover.verify(proof, verification_key_path=vkey_path)

print('verify(...) =>', verified)
print('snarkjs command:', captured['cmd'])
print('cwd:', captured['cwd'])
print('public JSON handed to snarkjs:', captured['public'])
print('proof JSON keys handed to snarkjs:', sorted(captured['proof'].keys()))


verify(...) => True
snarkjs command: ['npx', 'snarkjs', 'groth16', 'verify', '/home/runner/work/Olympus/Olympus/proofs/keys/verification_keys/document_existence_vkey.json', '/tmp/tmp58pqh3s4/public.json', '/tmp/tmp58pqh3s4/proof.json']
cwd: /home/runner/work/Olympus/Olympus/proofs
public JSON handed to snarkjs: ['12345', '0']
proof JSON keys handed to snarkjs: ['pi_a', 'pi_b', 'pi_c']
